# Data Analysis (Part 2)
All Dataframes using Pandas, Polars and PySpark will be named by "df", "dp" and "data", respectively.

In [17]:
# Import libraries
import polars as pl
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import pyspark
from pyspark.sql import SparkSession, SQLContext
from pyspark.sql.types import IntegerType
from pyspark.sql import functions as F
import polars.selectors as cs

In [2]:
# SparkSession
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Local")
    .master("local[*]") # For local mode with all available cores
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "1")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("--- Successful PySpark Configuration ---")
print(f"Execution Mode: {spark.conf.get('spark.master')}")
print(f"Driver Memory: {spark.conf.get('spark.driver.memory')}")
print(f"Executor Memory: {spark.conf.get('spark.executor.memory')}")

26/01/04 15:52:18 WARN Utils: Your hostname, ant resolves to a loopback address: 127.0.1.1; using 192.168.0.33 instead (on interface wlp1s0)
26/01/04 15:52:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/04 15:52:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


--- Successful PySpark Configuration ---
Execution Mode: local[*]
Driver Memory: 2g
Executor Memory: 2g


In [3]:
# Create DataFrame from CSV file
df = pd.read_csv('../data/games.csv',sep=',',header=0)
dp = pl.read_csv('../data/games.csv',separator=',',has_header=True)
data = spark.read.csv('../data/games.csv',header=True,inferSchema=True)

## Replace

In [14]:
# Replace values
df['platform'] = df['platform'].replace({'PS4_old':'PlayStation4_new', 'XOne_old':'Xbox One_new'})

dp = dp.with_columns(
    pl.col("platform").replace(
        {"PS4_old": "PlayStation4_new", "XOne_old": "Xbox One_new"},
        default=pl.col("platform") # Mantiene los valores que no coinciden
    )
)

from pyspark.sql import functions as F
mapping = {'PS4_old': 'PlayStation4_new', 'XOne_old': 'Xbox One_new'}
data = data.replace(to_replace=mapping, subset=['platform'])

/tmp/ipykernel_60475/1610530161.py:5: DeprecationWarning: The `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
  pl.col("platform").replace(


## Format

In [11]:
def format_content(element):
	'''Function to format data'''
	element = str(element).strip().lower().replace(" ","_").replace(",","")
	element = element.replace(";","").replace(".","")
	return element

In [12]:
# Change column names
df.columns = [format_content(col) for col in df.columns]
print(df.columns)

Index(['name', 'platform', 'year_of_release', 'genre', 'sales'], dtype='object')


In [13]:
# Change column names
dp.columns = [format_content(col) for col in dp.columns]
print(dp.columns)

['name', 'platform', 'year_of_release', 'genre', 'sales']


In [14]:
data = data.toDF(*[format_content(col) for col in data.columns])
print(data.columns)

['name', 'platform', 'year_of_release', 'genre', 'sales']


## Apply function
It is recommended to use native Polars and PySpark functions, and use custom functions only if it is extrictively necessary.

In [ ]:
# Format values in string columns
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].apply(format_content)

In [19]:
import polars.selectors as cs
dp = dp.with_columns(
    cs.string().str.strip_chars()
               .str.to_lowercase()
               .str.replace_all(" ", "_")
               .str.replace_all(r"[,;.]", "")
)
print(dp.head())

shape: (5, 5)
┌──────────────────────────┬──────────┬─────────────────┬──────────────┬───────┐
│ name                     ┆ platform ┆ year_of_release ┆ genre        ┆ sales │
│ ---                      ┆ ---      ┆ ---             ┆ ---          ┆ ---   │
│ str                      ┆ str      ┆ i64             ┆ str          ┆ f64   │
╞══════════════════════════╪══════════╪═════════════════╪══════════════╪═══════╡
│ wii_sports               ┆ wii      ┆ 2006            ┆ sports       ┆ 41.36 │
│ super_mario_bros         ┆ nes      ┆ 1985            ┆ platform     ┆ 29.08 │
│ mario_kart_wii           ┆ wii      ┆ 2008            ┆ racing       ┆ 15.68 │
│ wii_sports_resort        ┆ wii      ┆ 2009            ┆ sports       ┆ 15.61 │
│ pokemon_red/pokemon_blue ┆ gb       ┆ 1996            ┆ role-playing ┆ 11.27 │
└──────────────────────────┴──────────┴─────────────────┴──────────────┴───────┘


In [25]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Identificar columnas tipo string
string_cols = [
    field.name
    for field in data.schema.fields
    if isinstance(field.dataType, StringType)
]

# Aplicar transformaciones a todas las columnas string
data = data.select([
    F.regexp_replace(
        F.regexp_replace(
            F.lower(F.trim(F.col(c))),
            " ",
            "_"
        ),
        r"[,;.]",
        ""
    ).alias(c) if c in string_cols else F.col(c)
    for c in data.columns
])

data.show(5, truncate=False)

+------------------------+--------+---------------+------------+-----+
|name                    |platform|year_of_release|genre       |sales|
+------------------------+--------+---------------+------------+-----+
|wii_sports              |wii     |2006           |sports      |41.36|
|super_mario_bros        |nes     |1985           |platform    |29.08|
|mario_kart_wii          |wii     |2008           |racing      |15.68|
|wii_sports_resort       |wii     |2009           |sports      |15.61|
|pokemon_red/pokemon_blue|gb      |1996           |role-playing|11.27|
+------------------------+--------+---------------+------------+-----+
only showing top 5 rows



**Brief Comparative**
| Polars                           | PySpark                               |
| -------------------------------- | ------------------------------------- |
| `cs.string()`                    | Identificación manual de `StringType` |
| `.str.strip_chars()`             | `trim(col)`                           |
| `.str.to_lowercase()`            | `lower(col)`                          |
| `.str.replace_all(" ", "_")`     | `regexp_replace(col, " ", "_")`       |
| `.str.replace_all(r"[,;.]", "")` | `regexp_replace(col, r"[,;.]", "")`   |
| `with_columns()`                 | `select()` con lista de expresiones   |


In [ ]:
# Column operation
df['sales2'] = df['sales']*2

dp = dp.with_columns(
    (pl.col('sales') * 2).alias('sales2')
)
dp.head(3)

In [ ]:
data = data.withColumn('sales2', data['sales']*2)
data.show(3)